**-Setup**

In [0]:
from pyspark.sql.functions import(
    col,current_timestamp,lit,expr,coalesce,to_date
)
dbutils.widgets.text("storage_account","retailprojec1data")
dbutils.widgets.text("bronze_container","bronze")
dbutils.widgets.text("silver_container","silver")

storage_account=dbutils.widgets.get("storage_account")
bronze_container=dbutils.widgets.get("bronze_container")
silver_container=dbutils.widgets.get("silver_container")

base_bronze=f"abfss://{bronze_container}@{storage_account}.dfs.core.windows.net/"
base_silver=f"abfss://{silver_container}@{storage_account}.dfs.core.windows.net/"

bronze_transactions_path = f"{base_bronze}/transactions_delta/"
silver_dim_customers_path=f"{base_silver}/dim_customers/"
silver_dim_products_path=f"{base_silver}/dim_products/"
silver_fact_path=f"{base_silver}/fact_transactions/"
quarantine_path=f"{base_silver}/_quartine/fact_transactions/"

print(f"Bronze Transactions: {bronze_transactions_path}")
print(f"Silver dim_customers: {silver_dim_customers_path}")
print(f"silver dim_products: {silver_dim_products_path}")
print(f"silver Fact Target: {silver_fact_path}")
print(f"Quartine: {quarantine_path}")

**-Loading the sources**

In [0]:
bronze_txn=spark.read.format("delta").load(bronze_transactions_path)
print(f"Bronze transactions: {bronze_txn.count()}")

#silver dimensions - scd2 we are getting all the rows
dim_customers = spark.read.format("delta").load(silver_dim_customers_path)
print(f"dim_customers rows: {dim_customers.count()} (current: {dim_customers.filter(col('is_current')==True).count()})")

dim_products=spark.read.format("delta").load(silver_dim_products_path)
print(f"dim_products rows:{dim_products.count()} (current: {dim_products.filter(col('is_current')==True).count()})")


**-Parse transaction time**

In [0]:
txn_parsed = bronze_txn.withColumn("transaction_ts",coalesce(
    expr("try_to_timestamp(transaction_time, \"yyyy-MM-dd'T'HH:mm:ss\")"),
    expr("try_to_timestamp(transaction_time,'yyyy-MM-dd HH:mm:ss')"),
    expr("try_to_timestamp(transaction_time)")
)).withColumn("transaction_date",to_date(col("transaction_ts"))
              )

parse_failures = txn_parsed.filter(col("transaction_ts").isNull())
failure_count=parse_failures.count()

if failure_count>0:
    print(f"{failure_count} rows failed timestamp parsing - quaranting" )
    (parse_failures.withColumn("quarantine_reason",lit("transaction_time_parse_failed"))
     .withColumn("quarantine_ts",current_timestamp())
     .write.format("delta").mode("append").save(quarantine_path))
else:
    print("All rows parsed successfully")

clean_txn=txn_parsed.filter(col("transaction_ts").isNotNull())
print(f"Clean transactions: {clean_txn.count()}")



**-Join to dim_customers**

In [0]:
# Join transactions to dim_customers using point-in-time logic
# For each transaction, find the customer version whose effective period contains the transaction date

txn_with_customer = (clean_txn.alias("t")
    .join(
        dim_customers.alias("c"),
        (col("t.customer_id") == col("c.customer_id")) &
        (col("t.transaction_date") >= col("c.effective_from")) &
        (
            (col("t.transaction_date") < col("c.effective_to")) | 
            col("c.effective_to").isNull()  # is_current = true rows have NULL effective_to
        ),
        "left"
    )
    .select(
        col("t.*"),
        col("c.customer_sk"),
        col("c.first_name").alias("customer_first_name"),
        col("c.last_name").alias("customer_last_name"),
        col("c.loyalty_tier"),
        col("c.city").alias("customer_city"),
        col("c.state").alias("customer_state"),
    )
)

# Check for unmatched customers
unmatched_customers = txn_with_customer.filter(col("customer_sk").isNull()).count()
total_txn = txn_with_customer.count()
print(f"Transactions after customer join: {total_txn}")
print(f"Unmatched customers: {unmatched_customers}")

if unmatched_customers > 0:
    print(f"⚠️ {unmatched_customers} transactions have no matching customer dimension")
    print("   These will have NULL customer attributes in the fact table")

**-Join to dim_products**

In [0]:
# Join to dim_products using point-in-time logic
fact_enriched = (txn_with_customer.alias("t")
    .join(
        dim_products.alias("p"),
        (col("t.product_sku") == col("p.sku")) &
        (col("t.transaction_date") >= col("p.effective_from")) &
        (
            (col("t.transaction_date") < col("p.effective_to")) | 
            col("p.effective_to").isNull()
        ),
        "left"
    )
    .select(
        # Transaction fields
        col("t.transaction_id"),
        col("t.transaction_ts"),
        col("t.transaction_date"),
        col("t.store_id"),
        col("t.store_location"),
        col("t.register_id"),
        col("t.cashier_id"),
        col("t.payment_method"),
        col("t.status"),
        
        # Quantities and amounts
        col("t.quantity"),
        col("t.unit_price"),
        col("t.total_amount"),
        
        # Customer dimension (point-in-time)
        col("t.customer_id"),
        col("t.customer_sk"),
        col("t.customer_first_name"),
        col("t.customer_last_name"),
        col("t.loyalty_tier"),
        col("t.customer_city"),
        col("t.customer_state"),
        
        # Product dimension (point-in-time)
        col("t.product_sku"),
        col("p.product_sk"),
        col("p.product_name"),
        col("p.category"),
        col("p.retail_price").alias("catalog_price"),
        col("p.cost").alias("product_cost"),
    )
)

# Check for unmatched products
unmatched_products = fact_enriched.filter(col("product_sk").isNull()).count()
print(f"Fact rows after product join: {fact_enriched.count()}")
print(f"Unmatched products: {unmatched_products}")

**-Adding derived Columns**

In [0]:
fact_final = (fact_enriched
    # Revenue and margin calculations
    .withColumn("discount_amount",
        col("catalog_price") - col("unit_price")
    )
    .withColumn("gross_margin",
        (col("unit_price") - col("product_cost")) * col("quantity")
    )
    # Silver metadata
    .withColumn("silver_ingestion_ts", current_timestamp())
)

print(f"Final fact table columns: {len(fact_final.columns)}")
print(f"Final fact table rows: {fact_final.count()}")
fact_final.printSchema()

**-writing to silver path**

In [0]:
(fact_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_fact_path))

count = spark.read.format("delta").load(silver_fact_path).count()
print(f"✅ Silver fact_transactions written with {count} rows")

**-Verifying**

In [0]:
fact = spark.read.format("delta").load(silver_fact_path)

print("=" * 60)
print("fact_transactions_enriched Summary")
print("=" * 60)
print(f"Total rows:              {fact.count()}")
print(f"Distinct transactions:   {fact.select('transaction_id').distinct().count()}")
print(f"Distinct customers:      {fact.select('customer_id').distinct().count()}")
print(f"Distinct products:       {fact.select('product_sku').distinct().count()}")
print(f"Distinct stores:         {fact.select('store_id').distinct().count()}")
print(f"Date range:              {fact.agg({'transaction_date': 'min'}).collect()[0][0]} to {fact.agg({'transaction_date': 'max'}).collect()[0][0]}")

print(f"\nRevenue summary:")
from pyspark.sql.functions import sum as spark_sum, count, avg, round as spark_round

fact.filter(col("status") == "completed").agg(
    spark_round(spark_sum("total_amount"), 2).alias("total_revenue"),
    count("*").alias("total_transactions"),
    spark_round(avg("total_amount"), 2).alias("avg_transaction_value"),
    spark_round(spark_sum("gross_margin"), 2).alias("total_gross_margin"),
).show(truncate=False)

print("\nRevenue by store:")
(fact
    .filter(col("status") == "completed")
    .groupBy("store_id", "store_location")
    .agg(
        spark_round(spark_sum("total_amount"), 2).alias("revenue"),
        count("*").alias("txn_count"),
    )
    .orderBy(col("revenue").desc())
    .show(truncate=False))

print("\nRevenue by category:")
(fact
    .filter(col("status") == "completed")
    .groupBy("category")
    .agg(
        spark_round(spark_sum("total_amount"), 2).alias("revenue"),
        count("*").alias("txn_count"),
    )
    .orderBy(col("revenue").desc())
    .show(truncate=False))

print("\nRevenue by loyalty tier:")
(fact
    .filter(col("status") == "completed")
    .groupBy("loyalty_tier")
    .agg(
        spark_round(spark_sum("total_amount"), 2).alias("revenue"),
        count("*").alias("txn_count"),
    )
    .orderBy(col("revenue").desc())
    .show(truncate=False))

print("\nSample enriched rows:")
fact.select(
    "transaction_id", "transaction_date", "store_location",
    "customer_first_name", "loyalty_tier",
    "product_name", "category",
    "quantity", "unit_price", "total_amount", "gross_margin"
).show(10, truncate=False)